# 9.3 [실습] ReAct: 추론과 행동을 교차하기

> 9장 본문 코드를 절 단위로 추출. **맨 위 환경 설정부터 순서대로** 실행하세요.

## 환경 설정 — Colab에서 **맨 먼저** 실행
설치 후 "런타임 재시작" 안내가 떠도 무시하고 계속 실행하면 됩니다.

In [ ]:
!pip install langchain_openai==1.1.12 langchain_community==0.4.1 langchain==1.2.14 sqlalchemy numexpr pydantic tenacity nest_asyncio
!pip install -U duckduckgo_search==7.5.1 yfinance ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.4.2
    Uninstalling langgraph-sdk-0.4.2:
   

In [ ]:
import os
key=None
try:
    from google.colab import userdata
    key=userdata.get("OPENAI_API_KEY")
except Exception:
    pass
if not key:
    import getpass
    key=getpass.getpass("OpenAI API Key를 입력하세요: ")
os.environ["OPENAI_API_KEY"]=key

OpenAI API Key를 입력하세요: ··········


In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
search.invoke("현대자동차 최근 뉴스")

'현대자동차 현대차, GS25와 함께 ‘현차는 빵빵’ 아이스크림 출시 2026-06-28 현대자동차,기아 현대차·기아, 통학길 교통안전 지키는 ‘비전 펄스’ 기술 캠페인으로 칸 국제 광고제 수상 2026-06-28 현대자동차 현대자동차, 칸 라이언즈 2026 그랑프리 포함 2개 부문 수상 Jun 29, 2026 · 현대자동차그룹 뉴스룸 현대자동차그룹 뉴스룸의 기사와 이미지는 누구나 자유롭게 사용하실 수 있습니다. 그러나 현대자동차그룹 뉴스룸이 제공받은 일부 기사와 이미지는 사용에 제한이 있습니다. Jun 15, 2026 · 참가 제품/기술 2026.06.26 ‘현대 트랜스로컬 시리즈’ 신규 참여 4개 기관 발표 문화/예술 2026.06.23 현대자동차 구독 서비스 ‘현대 제네시스 셀렉션’, ‘360일 플랜’ 출시 기타/마케팅 2026.06.15 오프라인 전기차 체험 행사 개최 기타/마케팅 ... Google 뉴스을 (를) 사용하여 ‘현대자동차’ 주제에 관한 전체 기사를 읽고, 동영상을 보고, 다양한 콘텐츠를 탐색해 보세요. 조사대상 전 부문 1위 수상 (일반승용차, RV승용차, 경형승용차, 전기차, 프리미엄) 2025년 한국품질만족지수(KS-QEI) 조사 12개 부문 1위 (승용-준중형/중형/대형, SUV-소형/준중형/ 대형, 럭셔리-D,E세단/대형SUV, 전기차, 자동차AS, 스마트 모빌리티 서비스앱)'

In [ ]:
import yfinance as yf
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_openai import ChatOpenAI
from langchain_classic.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun

In [ ]:
# 1. 실제 금융 데이터 API 도구 (Yahoo Finance)
def get_financial_statement(symbol: str) -> str:
    """특정 기업의 티커를 입력받아 시가총액과 PER 등 재무 정보를 반환합니다."""
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info
        market_cap = info.get('marketCap', '정보 없음')
        forward_pe = info.get('forwardPE', '정보 없음')
        # 시가총액을 조 단위로 변환 (숫자일 경우)
        if isinstance(market_cap, (int, float)):
            market_cap = f"{market_cap / 1_000_000_000_000:.2f}조 원"
        return f"{symbol}의 시가총액은 {market_cap}이며, 추정 PER은 {forward_pe}입니다."
    except Exception as e:
        return f"재무 정보를 불러오는 데 실패했습니다: {e}"

In [ ]:
# 2. 실시간 뉴스 검색 도구 (DuckDuckGo)
search_tool = DuckDuckGoSearchRun()

# 3. 텍스트 요약 도구 (LLM을 도구 안에서 다시 호출)
def summarize_text(text: str) -> str:
    """검색된 긴 텍스트를 입력받아 핵심만 3줄로 요약합니다."""
    summary_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    response = summary_llm.invoke(f"다음 뉴스 텍스트를 3문장 이내로 핵심만 요약해:\n{text}")
    return response.content

In [ ]:
# 4. 도구 리스트 및 정교한 설명(Description) 설정
tools = [
    Tool(
        name="get_financial_statement",
        func=get_financial_statement,
        description="기업의 재무 정보(시가총액, PER 등)를 조회합니다. 한국 주식은 종목코드 뒤에 '.KS'를 붙이세요 (예: 현대차 -> 005380.KS)."
    ),
    Tool(
        name="get_latest_news",
        func=search_tool.run,
        description="기업 관련 최신 뉴스를 검색합니다."
    ),
    Tool(
        name="summarize_text",
        func=summarize_text,
        description="매우 긴 뉴스나 검색 결과를 요약할 때 사용합니다. 요약할 때는 반드시 이 도구를 사용하세요."
    )
]

In [ ]:
# 5. ReAct 표준 프롬프트
template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer in KOREAN to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

react_prompt = PromptTemplate.from_template(template)

In [ ]:
# 6. 모델 및 에이전트 실행기 구성
llm = ChatOpenAI(model="gpt-4o", temperature=0)
agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,
                               handle_parsing_errors=True)

# 7. 실행
user_query = "한국 주식 현대차(005380.KS)의 실제 재무 정보와, 최근 현대자동차 관련 뉴스를 검색해서 요약해줘."
response = agent_executor.invoke({"input": user_query})
print("\n[최종 답변]:\n", response["output"])



> Entering new AgentExecutor chain...
현대차의 재무 정보를 먼저 조회한 후, 최신 뉴스를 검색하여 요약하겠습니다.

Action: get_financial_statement
Action Input: "005380.KS"005380.KS의 시가총액은 131.44조 원이며, 추정 PER은 10.17174입니다.현대차의 재무 정보를 확인했습니다. 이제 현대자동차 관련 최신 뉴스를 검색하여 요약하겠습니다.

Action: get_latest_news
Action Input: "현대자동차"2 weeks ago - 현대자동차(現代自動車, 영어: Hyundai Motor Company, Hyundai Motors, HMC · )는 1967년 12월 29일에 설립된 현대자동차그룹 계열의 종합 자동차 제조 및 판매 업체이다. 본사는 서울특별시 서초구에 있다. 현대자동차는 정주영 현대그룹 ... April 19, 2026 - 현대자동차 차종 목록에서는 현대자동차가 현재 판매 중인 차량, 엔진과 과거에 판매했던 차량 그리고 컨셉트카를 목록으로 정리했다. 1 day ago - Hyundai Motor Company, often referred to as Hyundai Motors (Korean: 현대자동차 August 27, 2025 - 환경기술연구소는 경기도 용인시 기흥구 마북동에 있으며 현대차·기아의 기술연구소 중 하나이다. 이 연구소는 마북연구소라고 불리며, 현대자동차그룹의 수소전기자동차 연구 개발에 집중하고 있다. 1 week ago - 현대자동차그룹(Hyundai Motor Group)은 그룹의 대표 사업이자 완성차 브랜드 현대자동차, 기아, 제네시스를 필두로 한 재벌로 2023년 현재 자산총액 기준 대한민국 재계 서열 3위에 위치한다.[4] ...Thought: 최신 뉴스가 여러 개 검색되었으므로, 이를 요약하여 제공하겠습니다.

Action: summarize_text
Action Input: "2 weeks ago - 현대자동차(現代